In [15]:
import fitz
import re

## pdf의 header와 footer를 제거하기 위한 ROI(Region Of Interest) 정의
def set_RoI(rect, content_area):
    
    (x0, y0, x1, y1) = rect
    (x_ratio, y_ratio) = content_area
    
    w = (x1-x0)
    h = (y1-y0)
    
    new_w = w * x_ratio
    new_h = h * y_ratio
    
    rect.x0 = x0 + (w - new_w)/2
    rect.y0 = y0 + (h - new_h)/2
    rect.x1 = x1 - (w - new_w)/2
    rect.y1 = y1 - (h - new_h)/2    
    
    return rect

## pdf 파일에 있는 text를 추출하는 함수
def pdf2text(pdf_path, content_area):
    
    full_text = list()

    ## PDF 파일 open
    with fitz.open(pdf_path) as pdf:

        ## PDF 페이지 별로 정보 조회
        for page_num in range(len(pdf)):

            page = pdf.load_page(page_num)            
            rect = page.rect            
            
            ## header, footer 제거
            text = page.get_text(clip=set_RoI(rect, content_area))

            
            ## table 제거
            tables = page.find_tables().tables            
            for _tab in tables:
                # print(_tab)
                text = text.replace(page.get_textbox(_tab.bbox), "")     
                
            # 표, 이미지 제목 제거
            pattern_figure = r"\bFigure\b\s\d+\-\d+\.\s[A-Z][a-z0-9\-\’\(\)\,\s\bIPB\b\bMDMP\b\bEarth\b]+"
            pattern_table = r"\bTable\b\s\d+\-\d+\.\s[A-Z][a-z0-9\-\’\(\)\,\s\bIPB\b\bArmy\b\bASCOPE\b\bPMESII\b\bCARVER\b]+"
            find_title_figure = re.findall(pattern_figure, text)
            find_title_table= re.findall(pattern_table, text)
            
  
            for figure_title in find_title_figure:
                if figure_title is not None:
                    text = text.replace(figure_title, "")

                    
            for table_title in find_title_table:
                if table_title is not None:
                    text = text.replace(table_title, "")

            # 챕터 제목 제거
            pattern_chapter = r"\bChapter\b\s\d+\s\n+[0-9a-zA-Z\s\—]+\s\n+(?=[A-Z])"
            find_chapter = re.findall(pattern_chapter, text)
            for chapter_title in find_chapter:
                if chapter_title is not None:
                    text = text.replace(chapter_title, "")

            # 정제된 내용 추가
            full_text.append(text)

    return full_text

## PDF 파일 경로 지정
pdf_path = "IPB.pdf"
 
## header, footer 지우기 위한 RoI 설정
x_clip_ratio = 1.0
y_clip_ratio = 0.85

## header, footer, 표 정보가 제거된 페이지(index) 별 텍스트 추출
contents = pdf2text(pdf_path, (x_clip_ratio, y_clip_ratio))

In [16]:
import re

def split_by_custom_pattern(text, paragraph_pattern, spliter_pattern):

    result = list()

    ## 문단에 대한 
    matches = list(re.finditer(paragraph_pattern, text))

    if not matches:
        return result
    
    for i in range(len(matches)):
        #start = matches[i].start()
        end = matches[i].end()
        
        # 현재 match와 다음 match 사이의 내용을 추출
        if i < len(matches) - 1:
            next_start = matches[i + 1].start()
            content = text[end:next_start].strip()
        else:
            content = text[end:].strip()
        
        _splited_obj = matches[i].group()
        _target_list = re.findall(spliter_pattern, _splited_obj)
        print(_target_list)
        # if len(_target_list) == 0:
        #     continue
        
        # if len(_target_list[0].strip()) == 0:
        #     print(_target_list, content)

        # 패턴을 key로, 내용을 value로 저장
        result.append((_target_list[0].strip(), content))
    # print(result)
    return result

In [17]:
# from tqdm import tqdm
import pandas as pd
"""
### IPB 페이지 정리
- 1) PART ONE : Fundamental Principles, Process Activities, and Relationships - 15p ~ 36p
- 2) PART TWO Fundamental Task Techniques - 37p ~ 120p
- 3) PART THREE Considerations for Operations and Environments - 121p ~ 166p
"""


pdf = dict()
pdf["PART ONE"] = contents[14:36]
pdf["PART TWO"] = contents[36:120]
pdf["PART THREE"] = contents[120:166]

## 컨텐츠 내 유사한 정보를 포함하고 있는 정보 단위 패턴 정의
# 수정 후
paragraph_pattern = r'\n[^a-z0-9\.\n]+\n*[^a-z0-9\.\n]*\n*[^a-z0-9\.\n]*\n\d+-\d+\.\s'
spliter_pattern = r'\n[^a-z0-9\.\n]+\n*[^a-z0-9\.\n]*\n*[^a-z0-9\.\n]*'

## pandas dataframe로 만들 컬럼 정보 초기화
result = dict()
result["title"] = list()
result["sub_title"] = list()
result["content"] = list()
for title, target_contents in pdf.items():

    main_contents = '\n'.join(target_contents)
    total_contents = [(_sub_title, _content.replace("\n"," ").strip()) for _sub_title, _content in split_by_custom_pattern(main_contents, paragraph_pattern=paragraph_pattern, spliter_pattern=spliter_pattern)]
    for _idx, (_sub_title, _content) in enumerate(total_contents):
        while True:
            prev_length = len(_content)
            _content = _content.replace("  ", " ")
            if len(_content) == prev_length:
                total_contents[_idx] = (_sub_title, _content)
                break

    for _sub_title, _content in total_contents:
        result["title"].append(title)
        result["sub_title"].append(_sub_title)
        result["content"].append(_content)


df_master = pd.DataFrame(data=result)

['\nINTELLIGENCE PREPARATION OF THE BATTLEFIELD (IPB) \nDEFINED \n']
['\nIPB PROCESS ACTIVITIES \n']
['\nSTAFF COLLABORATION \n']
['\nRELATIONSHIPS \n']
['\nTARGETING\n']
['\nRISK MANAGEMENT \n']
['\nINFORMATION COLLECTION \n']
['\nGENERATE INTELLIGENCE KNOWLEDGE \n']
['\nSITUATION DEVELOPMENT\n']
['\nMULTI-DOMAIN UNDERSTANDING OF THE OPERATIONAL \nENVIRONMENT \n']
['\nIMPORTANCE OF DOMAIN INTERDEPENDENCE \n']
['\nOPERATIONAL FRAMEWORK CONSIDERATIONS \n']
['\nHOLISTIC VIEW OF THE OPERATIONAL ENVIRONMENT \n']
['\nIPB AND THE ARMY’S STRATEGIC ROLES \n']
['\nIPB AND PLANNING \n']
['\nMILITARY DECISION-MAKING PROCESS\n']
['\nTROOP LEADING PROCEDURES \n']
['\nIPB AND DECISION MAKING \n']
['\nWHAT IS IT? \n']
['\nSO WHAT? \n']
['\nHOW TO DO IT: THE PROCESS \n']
['\nIDENTIFY THE LIMITS OF THE COMMANDER’S AREA OF \nOPERATIONS \n']
['\nNTIFY THE LIMITS OF THE COMMANDER’S AREA OF \nINTEREST \n']
['\nIDENTIFY SIGNIFICANT CHARACTERISTICS OF THE AREA OF \nOPERATIONS AND AREA OF INTEREST FOR FURTHER

In [18]:
## SECTION에 대한 전반적인 설명 제외
# df_master = df_master.loc[~df_master["sub_title"].str.contains("SECTION")]
df_master.to_excel("./IPB_전처리(최종)_v4.1.xlsx", index=False)